# Differential Machine Learning — PyTorch

Воспроизведение методологии Huge & Savine на PyTorch: обучение по значениям payoff и pathwise-производным для **Black–Scholes** и **Bachelier** (размерности 1, 7, 50). Реализация не требует TensorFlow.

Используются исходные генераторы LSM с antithetic sampling, нормализация производных, `Softplus`, Adam и one-cycle learning-rate schedule. Исходный PyTorch-порт адаптирован для самостоятельного воспроизводимого запуска.

Работа: [Differential Machine Learning](https://arxiv.org/abs/2005.02347).

## Настройки

`RUN_MODE = "quick"` — короткая проверка; `"full"` — 100 эпох и размеры выборок из демонстрационного ноутбука. На Colab автоматически выбирается CUDA при наличии GPU. Для сопоставимости можно принудительно указать `DEVICE_NAME = "cpu"`.

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

RUN_MODE = "quick"          # "quick" / "full"
DEVICE_NAME = "auto"       # "auto" / "cpu" / "cuda"
SEED = 20261009
DTYPE = "float32"          # "float32" / "float64"
HIDDEN_UNITS = 20
HIDDEN_LAYERS = 4
LAM = 1.0
N_TEST = 1000 if RUN_MODE == "quick" else 4096
EPOCHS = 15 if RUN_MODE == "quick" else 100
EPS = 1e-8

assert RUN_MODE in {"quick", "full"}
assert DEVICE_NAME in {"auto", "cpu", "cuda"}
assert DTYPE in {"float32", "float64"}
if DEVICE_NAME == "auto":
    DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE_NAME == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA не обнаружена. Выбери CPU или включи GPU в Colab.")
device = torch.device(DEVICE_NAME)
real_type = torch.float32 if DTYPE == "float32" else torch.float64
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"PyTorch {torch.__version__}; device={device}; dtype={DTYPE}; mode={RUN_MODE}")

## Модель и производные по входам

In [ ]:
def vanilla_net(input_dim, hidden_units, hidden_layers, seed=None):
    if seed is not None:
        torch.manual_seed(seed)

    layers = [nn.Linear(input_dim, hidden_units), nn.Softplus()]
    for _ in range(hidden_layers - 1):
        layers += [nn.Linear(hidden_units, hidden_units), nn.Softplus()]
    layers += [nn.Linear(hidden_units, 1)]

    model = nn.Sequential(*layers).to(device=device, dtype=real_type)

    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    return model


In [ ]:
def twin_forward(model, x):
    """Forward + d(y)/d(x) per sample (diagonal of the Jacobian)."""
    x = x.detach().clone().requires_grad_(True)
    y = model(x)
    dydx = torch.autograd.grad(
        outputs=y,
        inputs=x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
    )[0]
    return y, dydx


## Нормализация и обучение

Функция потерь differential network: `alpha * MSE(y_pred, y) + beta * MSE(lambda_j * dydx_pred, lambda_j * dydx)`; `alpha=1/(1+lam*n)` и `beta=1-alpha`. Для производных используется `create_graph=True`.

In [ ]:
def normalize_data(x_raw, y_raw, dydx_raw=None, crop=None):
    m = crop if crop is not None else x_raw.shape[0]

    xc = x_raw[:m]; yc = y_raw[:m]
    dyc = dydx_raw[:m] if dydx_raw is not None else None

    x_mean = xc.mean(axis=0); x_std = xc.std(axis=0) + EPS
    x = (xc - x_mean) / x_std

    y_mean = yc.mean(axis=0); y_std = yc.std(axis=0) + EPS
    y = (yc - y_mean) / y_std

    if dyc is not None:
        dy_dx = dyc / y_std * x_std
        lambda_j = 1.0 / np.sqrt((dy_dx ** 2).mean(axis=0)).reshape(1, -1)
    else:
        dy_dx = None; lambda_j = None

    return x_mean, x_std, x, y_mean, y_std, y, dy_dx, lambda_j


In [ ]:
class Neural_Approximator:

    def __init__(self, x_raw, y_raw, dydx_raw=None):
        self.x_raw, self.y_raw, self.dydx_raw = x_raw, y_raw, dydx_raw
        self.model = None; self.optimizer = None
        self.m = self.n = 0
        self.differential = False
        self.weight_seed = None
        self.hidden_units = None
        self.hidden_layers = None

    def _make_model(self):
        self.model = vanilla_net(self.n, self.hidden_units,
                                 self.hidden_layers, self.weight_seed)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-8)

    def prepare(self, m, differential, lam=1.0,
                hidden_units=20, hidden_layers=4, weight_seed=None):

        (self.x_mean, self.x_std, self.x,
         self.y_mean, self.y_std, self.y,
         self.dy_dx, self.lambda_j) = normalize_data(
            self.x_raw, self.y_raw, self.dydx_raw, m)

        self.differential = differential
        self.m, self.n = self.x.shape
        self.hidden_units, self.hidden_layers = hidden_units, hidden_layers
        self.weight_seed = weight_seed

        self._make_model()

        if differential:
            if self.dy_dx is None:
                raise Exception("No differential labels for differential training")
            self.alpha = 1.0 / (1.0 + lam * self.n)
            self.beta  = 1.0 - self.alpha
            self.lambda_t = torch.tensor(self.lambda_j,
                                         dtype=real_type, device=device)

        self.x_t = torch.tensor(self.x, dtype=real_type, device=device)
        self.y_t = torch.tensor(self.y, dtype=real_type, device=device)
        if differential:
            self.dy_dx_t = torch.tensor(self.dy_dx,
                                        dtype=real_type, device=device)

    def _vanilla_epoch(self, lr, batch_size):
        for g in self.optimizer.param_groups: g['lr'] = lr
        first, last = 0, min(batch_size, self.m)
        self.model.train()
        while first < self.m:
            xb, yb = self.x_t[first:last], self.y_t[first:last]
            self.optimizer.zero_grad()
            loss = F.mse_loss(self.model(xb), yb)
            loss.backward(); self.optimizer.step()
            first, last = last, min(last + batch_size, self.m)

    def _diff_epoch(self, lr, batch_size):
        for g in self.optimizer.param_groups: g['lr'] = lr
        first, last = 0, min(batch_size, self.m)
        self.model.train()
        while first < self.m:
            xb = self.x_t[first:last]
            yb = self.y_t[first:last]
            db = self.dy_dx_t[first:last]

            self.optimizer.zero_grad()
            y_pred, dydx_pred = twin_forward(self.model, xb)

            loss_val  = F.mse_loss(y_pred, yb)
            loss_diff = F.mse_loss(dydx_pred * self.lambda_t, db * self.lambda_t)
            loss = self.alpha * loss_val + self.beta * loss_diff

            loss.backward(); self.optimizer.step()
            first, last = last, min(last + batch_size, self.m)

    def train(self, description="training", reinit=True, epochs=100,
              learning_rate_schedule=None,
              batches_per_epoch=16, min_batch_size=256,
              callback=None, callback_epochs=None):

        if learning_rate_schedule is None:
            learning_rate_schedule = [(0.0, 1e-8), (0.2, 0.1),
                                      (0.6, 0.01), (0.9, 1e-6), (1.0, 1e-8)]
        if callback_epochs is None:
            callback_epochs = []
        batch_size = max(min_batch_size, self.m // batches_per_epoch)
        lr_epochs, lr_rates = zip(*learning_rate_schedule)

        if reinit:
            self._make_model()

        if callback and 0 in callback_epochs:
            callback(self, 0)

        for epoch in tqdm(range(epochs), desc=description):
            lr = float(np.interp(epoch / epochs, lr_epochs, lr_rates))
            if not self.differential:
                self._vanilla_epoch(lr, batch_size)
            else:
                self._diff_epoch(lr, batch_size)
            if callback and epoch in callback_epochs:
                callback(self, epoch)

        if callback and epochs in callback_epochs:
            callback(self, epochs)

    @torch.no_grad()
    def predict_values(self, x):
        xs = (x - self.x_mean) / self.x_std
        xt = torch.as_tensor(xs, dtype=real_type, device=device)
        self.model.eval()
        ys = self.model(xt).cpu().numpy()
        return self.y_mean + self.y_std * ys

    def predict_values_and_derivs(self, x):
        xs = (x - self.x_mean) / self.x_std
        xt = torch.as_tensor(xs, dtype=real_type, device=device)
        self.model.eval()
        ys, ds = twin_forward(self.model, xt)
        ys = ys.detach().cpu().numpy(); ds = ds.detach().cpu().numpy()
        y    = self.y_mean + self.y_std * ys
        dydx = self.y_std / self.x_std * ds
        return y, dydx


## Генераторы LSM и аналитические формулы цен / дельт

In [ ]:
def bsPrice(spot, strike, vol, T):
    d1 = (np.log(spot / strike) + 0.5 * vol * vol * T) / vol / np.sqrt(T)
    d2 = d1 - vol * np.sqrt(T)
    return spot * norm.cdf(d1) - strike * norm.cdf(d2)

def bsDelta(spot, strike, vol, T):
    d1 = (np.log(spot / strike) + 0.5 * vol * vol * T) / vol / np.sqrt(T)
    return norm.cdf(d1)

def bsVega(spot, strike, vol, T):
    d1 = (np.log(spot / strike) + 0.5 * vol * vol * T) / vol / np.sqrt(T)
    return spot * np.sqrt(T) * norm.pdf(d1)


class BlackScholes:
    def __init__(self, vol=0.2, T1=1, T2=2, K=1.10, volMult=1.5):
        self.spot = 1
        self.vol, self.T1, self.T2, self.K, self.volMult = vol, T1, T2, K, volMult

    def trainingSet(self, m, anti=True, seed=None):
        rng = np.random.default_rng(seed)
        returns = rng.normal(size=[m, 2])

        vol0 = self.vol * self.volMult
        R1 = np.exp(-0.5 * vol0 * vol0 * self.T1
                    + vol0 * np.sqrt(self.T1) * returns[:, 0])
        R2 = np.exp(-0.5 * self.vol * self.vol * (self.T2 - self.T1)
                    + self.vol * np.sqrt(self.T2 - self.T1) * returns[:, 1])
        S1 = self.spot * R1; S2 = S1 * R2
        pay = np.maximum(0, S2 - self.K)

        if anti:
            R2a = np.exp(-0.5 * self.vol * self.vol * (self.T2 - self.T1)
                         - self.vol * np.sqrt(self.T2 - self.T1) * returns[:, 1])
            S2a = S1 * R2a
            paya = np.maximum(0, S2a - self.K)
            X, Y = S1, 0.5 * (pay + paya)
            Z1 = np.where(S2  > self.K, R2,  0.0).reshape((-1, 1))
            Z2 = np.where(S2a > self.K, R2a, 0.0).reshape((-1, 1))
            Z = 0.5 * (Z1 + Z2)
        else:
            X, Y = S1, pay
            Z = np.where(S2 > self.K, R2, 0.0).reshape((-1, 1))

        return X.reshape(-1, 1), Y.reshape(-1, 1), Z.reshape(-1, 1)

    def testSet(self, lower=0.35, upper=1.65, num=100, seed=None):
        spots  = np.linspace(lower, upper, num).reshape((-1, 1))
        prices = bsPrice(spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        deltas = bsDelta(spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        vegas  = bsVega (spots, self.K, self.vol, self.T2 - self.T1).reshape((-1, 1))
        return spots, spots, prices, deltas, vegas


In [ ]:
def bachPrice(spot, strike, vol, T):
    d = (spot - strike) / vol / np.sqrt(T)
    return vol * np.sqrt(T) * (d * norm.cdf(d) + norm.pdf(d))

def bachDelta(spot, strike, vol, T):
    d = (spot - strike) / vol / np.sqrt(T)
    return norm.cdf(d)

def bachVega(spot, strike, vol, T):
    d = (spot - strike) / vol / np.sqrt(T)
    return np.sqrt(T) * norm.pdf(d)


def genCorrel(n, rng):
    randoms = rng.uniform(low=-1.0, high=1.0, size=(2 * n, n))
    cov = randoms.T @ randoms
    invvols = np.diag(1.0 / np.sqrt(np.diagonal(cov)))
    return np.linalg.multi_dot([invvols, cov, invvols])


class Bachelier:
    def __init__(self, n, T1=1, T2=2, K=1.10, volMult=1.5):
        self.n = n
        self.T1, self.T2, self.K, self.volMult = T1, T2, K, volMult

    def trainingSet(self, m, anti=True, seed=None, bktVol=0.2):
        rng = np.random.default_rng(seed)

        self.S0 = np.repeat(1.0, self.n)
        self.corr = genCorrel(self.n, rng)

        self.a = rng.uniform(low=1.0, high=10.0, size=self.n)
        self.a /= np.sum(self.a)
        vols = rng.uniform(low=5.0, high=50.0, size=self.n)

        avols = (self.a * vols).reshape((-1, 1))
        v = np.sqrt(np.linalg.multi_dot([avols.T, self.corr, avols]).reshape(1))
        self.vols = vols * bktVol / v
        self.bktVol = bktVol

        diagv = np.diag(self.vols)
        self.cov = np.linalg.multi_dot([diagv, self.corr, diagv])
        self.chol  = np.linalg.cholesky(self.cov) * np.sqrt(self.T2 - self.T1)
        self.chol0 = self.chol * self.volMult * np.sqrt(self.T1 / (self.T2 - self.T1))

        normals = rng.normal(size=[2, m, self.n])
        inc0 = normals[0] @ self.chol0.T
        inc1 = normals[1] @ self.chol.T

        S1 = self.S0 + inc0; S2 = S1 + inc1
        bkt2 = S2 @ self.a
        pay  = np.maximum(0, bkt2 - self.K)

        if anti:
            S2a = S1 - inc1
            bkt2a = S2a @ self.a
            paya = np.maximum(0, bkt2a - self.K)
            X, Y = S1, 0.5 * (pay + paya)
            Z1 = np.where(bkt2  > self.K, 1.0, 0.0).reshape((-1, 1)) * self.a.reshape((1, -1))
            Z2 = np.where(bkt2a > self.K, 1.0, 0.0).reshape((-1, 1)) * self.a.reshape((1, -1))
            Z = 0.5 * (Z1 + Z2)
        else:
            X, Y = S1, pay
            Z = np.where(bkt2 > self.K, 1.0, 0.0).reshape((-1, 1)) * self.a.reshape((1, -1))

        return X, Y.reshape(-1, 1), Z

    def testSet(self, lower=0.5, upper=1.50, num=4096, seed=None):
        rng = np.random.default_rng(seed)
        adj = 1 + 0.5 * np.sqrt((self.n - 1) * (upper - lower) / 12)
        adj_lower = 1.0 - (1.0 - lower) * adj
        adj_upper = 1.0 + (upper - 1.0) * adj
        spots = rng.uniform(low=adj_lower, high=adj_upper, size=(num, self.n))

        baskets = (spots @ self.a).reshape((-1, 1))
        prices  = bachPrice(baskets, self.K, self.bktVol, self.T2 - self.T1).reshape((-1, 1))
        deltas  = bachDelta(baskets, self.K, self.bktVol, self.T2 - self.T1) @ self.a.reshape((1, -1))
        vegas   = bachVega (baskets, self.K, self.bktVol, self.T2 - self.T1)
        return spots, baskets, prices, deltas, vegas


## Эксперименты

Каждая пара `standard`/`differential` обучается на **одних и тех же payoff-примерах**. Нейросети стартуют с одного seed; оценка — на отдельном тестовом наборе относительно аналитических цен и производных. RMSE дельт усредняется по всем координатам (включая d=50).

In [ ]:
SIZES = ({
    "Black–Scholes": [1024],
    "Bachelier d=1": [1024],
    "Bachelier d=7": [4096],
    "Bachelier d=50": [4096],
} if RUN_MODE == "quick" else {
    "Black–Scholes": [1024, 8192],
    "Bachelier d=1": [1024, 8192],
    "Bachelier d=7": [4096, 8192, 16384],
    "Bachelier d=50": [16384, 65536, 131072],
})

def evaluate_task(task, sizes, task_index):
    gen = BlackScholes() if task == "Black–Scholes" else Bachelier(int(task.split('=')[1]))
    # Отдельный, детерминированный поток случайности на каждую задачу.
    x, y, z = gen.trainingSet(max(sizes), anti=True, seed=SEED + 101 * task_index)
    xt, _, yt, zt, _ = gen.testSet(num=N_TEST, seed=SEED + 1000 + 101 * task_index)
    rows = []
    for m in sizes:
        for method in ("standard", "differential"):
            approximator = Neural_Approximator(x, y, z)
            approximator.prepare(m, differential=(method == "differential"),
                                 lam=LAM, hidden_units=HIDDEN_UNITS,
                                 hidden_layers=HIDDEN_LAYERS,
                                 weight_seed=SEED + 10_000 + task_index * 100 + m)
            start = time.perf_counter()
            approximator.train(f"{task} | N={m} | {method}", epochs=EPOCHS)
            if device.type == "cuda":
                torch.cuda.synchronize()
            seconds = time.perf_counter() - start
            yp, zp = approximator.predict_values_and_derivs(xt)
            price_rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
            delta_rmse = float(np.sqrt(np.mean((zp - zt) ** 2)))
            rows.append({"task": task, "train_size": m, "method": method,
                         "rmse_price": price_rmse, "rmse_delta": delta_rmse,
                         "seconds": seconds})
            print(f"{task:16} N={m:6} {method:12} price={price_rmse:.6g} delta={delta_rmse:.6g} time={seconds:.1f}s")
    return rows

results = []
for i, (task, sizes) in enumerate(SIZES.items()):
    results.extend(evaluate_task(task, sizes, i))
results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("dml_pytorch_results.csv", index=False)
print("Сохранено: dml_pytorch_results.csv")

## Итог: преимущество Differential ML и графики

In [ ]:
price = results_df.pivot(index=["task", "train_size"], columns="method", values="rmse_price")
delta = results_df.pivot(index=["task", "train_size"], columns="method", values="rmse_delta")
gain = pd.DataFrame({"price_gain_x": price["standard"] / price["differential"],
                     "delta_gain_x": delta["standard"] / delta["differential"]})
display(gain.round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for task in SIZES:
    sub = gain.loc[task]
    axes[0].plot(sub.index, sub["price_gain_x"], marker="o", label=task)
    axes[1].plot(sub.index, sub["delta_gain_x"], marker="o", label=task)
for ax, label in zip(axes, ["RMSE цены: standard / differential", "RMSE дельты: standard / differential"]):
    ax.set_xscale("log", base=2)
    ax.axhline(1, linestyle="--", linewidth=1)
    ax.set_xlabel("Размер обучающей выборки")
    ax.set_ylabel("Выигрыш, ×")
    ax.set_title(label)
    ax.grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

## Примечания к воспроизводимости

- Этот ноутбук — **самостоятельная реализация на PyTorch**. Сравнение с TensorFlow и результаты CPU/GPU приведены в `README.md`; оно не требуется для запуска.
- Генерация Monte Carlo использует `np.random.default_rng`, поэтому **одинаковые seed не гарантируют побитовое совпадение** с оригинальным ноутбуком TensorFlow, использующим другой генератор.
- Инициализация сети — `xavier_uniform_` из предоставленного PyTorch-порта; это не идентичная инициализация исходного TensorFlow. Математическая корректность и результаты при общих весах проверялись отдельно.
- `full` воспроизводит выбранные размеры и параметры обучения, но не все протоколы оценки/валидации, представленные в научной статье.
- Для установки библиотек используй `requirements.txt`. В Colab PyTorch может быть уже установлен.